# 02 - Modeling

Training and test runs. First we import, load, and split the dataset. `.pkl` and `.csv` files are svaed to `results/`

In [ ]:
%load_ext autoreload
%autoreload 2

import pickle
import time
import warnings

import numpy as np
import pandas as pd

import lightgbm as lgb

from inf_minesweeper.loader import load, split_by_game
from inf_minesweeper.models import (Constant,                   # baseline: constant mine density
                                    LocalRatio,                 # baseline: average no. of mines around candidate cell
                                    LogisticFeatures,           # baseline/model: logistic regression
                                    GradientBoosted,            # model: LightGBM (PyPI v4.7.0)
                                    )
from inf_minesweeper.metrics import evaluate

d = load("../data/raw/guesses.npz")
W, y, q, game, pos = d["window"], d["mine"], d["q"], d["game"], d["position"]

train, test = split_by_game(game)                       # 4,000 / 1,000 games
# hold out some training games for validation
fit, val = split_by_game(np.where(train, game, -1), test_frac=0.125, seed=1)
fit, val = fit & train, val & train

baseline = Constant().fit(W[train], y[train]).predict(W[test])
results = {}


def score(name, p):
    results[name] = evaluate(y[test], p, q[test], game[test], pos[test], baseline)
    return pd.DataFrame(results).T.round(3)

## Baselines

In [ ]:
for name, model in {
    "constant, density 0.21": Constant(0.21),
    "constant, mine rate": Constant(),
    "local ratio": LocalRatio(),
    "logistic, 8 features": LogisticFeatures(),
}.items():
    score(name, model.fit(W[train], y[train]).predict(W[test]))
pd.DataFrame(results).T.round(3)

## Gradient-boosted trees

In [ ]:
trained = {}
for k in (3, 5, 7):
    start = time.time()
    model = GradientBoosted(k=k).fit(W[fit], y[fit], val=(W[val], y[val]))
    score(f"LightGBM {k}x{k}", model.predict(W[test]))
    trained[f"LightGBM {k}x{k}"] = model
    with open(f"../results/lgb{k}.pkl", "wb") as f:
        pickle.dump(model, f)
    print(f"k = {k}: {model.n_trees} trees, {(time.time() - start):.2f}s; lgb{k}.pkl saved")

table = pd.DataFrame(results).T.round(3)
table.to_csv("../results/model_results.csv")
print(f"model_results.csv saved")
display(table)

## Convolutional Neural Net

meow

## Sensitivity test

Perturbing the mine density by $\pm 0.01$ will verify the value and highlight the potential for distribution shift.

In [ ]:
shift = {}
for p, path in [(0.20, "../data/raw/shift_p020.npz"), (0.22, "../data/raw/shift_p022.npz")]:
    s = load(path)
    base = np.full(len(s["q"]), y[train].mean())        # same constant as on the test set
    for name, model in trained.items():                 # dict of name -> fitted model
        p_hat = model.predict(s["window"])
        shift[(name, p)] = evaluate(s["mine"], p_hat, s["q"], s["game"], s["position"], base)

pd.DataFrame(shift).T.round(3)